# 02 - UCI Heart Disease: data preprocessing

**MedIntel AI | Research prototype**

Prepare the same official processed Cleveland cohort used in
`01_data_exploration.ipynb`. This notebook runs independently: it loads the raw
source, prepares a binary target, creates a reproducible training/test split,
and builds and verifies preprocessing. **No prediction model is trained.**

## Workflow

1. Load and validate the source without learning any transformation.
2. Separate the diagnosis from the 13 input features.
3. Split raw features and labels into training and test sets.
4. Fit imputation, scaling, and encoding on training features only.
5. Apply the fitted transformations to the test features without refitting.
6. Check alignment, missingness, and the learned preprocessing statistics.

## Sources and setup

- [Official UCI dataset](https://archive.ics.uci.edu/dataset/45/heart+disease)
  and [source file](https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data).
- Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989).
  *Heart Disease* [Dataset]. UCI Machine Learning Repository.
  [DOI: 10.24432/C52P4X](https://doi.org/10.24432/C52P4X).
  Cleveland data collection: Robert Detrano, Cleveland Clinic Foundation.
  UCI lists the dataset under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- [scikit-learn: preventing data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

Install the project's `requirements.txt`, select its virtual environment in
VS Code or Jupyter, and run all cells in order. Start Jupyter from the project
root or `notebooks/`. The existing file in `data/raw/` is reused; only a missing
file requires a download. No new dependencies are needed.

## 1. Import the preprocessing tools

**Code explanation:** pandas manages tables and NumPy supports numerical checks.
`train_test_split` creates the holdout. `SimpleImputer`, `StandardScaler`, and
`OneHotEncoder` provide the transformations; `ColumnTransformer` assigns them
to feature groups and `Pipeline` keeps their order explicit. `clone` produces
an unfitted copy so we can keep a reusable template. Print package versions
alongside the results for reproducibility. None of these imports fits anything.

In [1]:
from hashlib import sha256
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print(f"pandas {pd.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

pandas 3.0.6 | NumPy 2.5.4 | scikit-learn 1.9.1


## 2. Load the unmodified official source

**Code explanation:** Locate the project from the current directory or its
parents and reuse `data/raw/processed.cleveland.data`. If absent, download it
directly from UCI with a timeout. Supply the documented column order because
this file has no header. Interpret `?` as missing rather than treating it as
a category. Parsing missing markers is deterministic; no imputation occurs here.
Check the expected dimensions and print a SHA-256 fingerprint of the file.
The source is never overwritten when it already exists.

In [2]:
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "heart-disease/processed.cleveland.data"
)
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]
working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder for folder in [working_directory, *working_directory.parents]
        if (folder / "requirements.txt").is_file()
        and (folder / "data" / "raw").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from MedIntel-AI or its notebooks folder.")

raw_path = PROJECT_ROOT / "data" / "raw" / "processed.cleveland.data"
if not raw_path.exists():
    try:
        with urlopen(DATA_URL, timeout=30) as response:
            source_bytes = response.read()
    except (URLError, TimeoutError) as error:
        raise RuntimeError(
            "Could not reach UCI. Check your connection or save DATA_URL to "
            "data/raw/processed.cleveland.data, then rerun this cell."
        ) from error
    raw_path.write_bytes(source_bytes)

df = pd.read_csv(raw_path, header=None, names=COLUMNS, na_values=["?"])
if df.shape != (303, 14):
    raise ValueError(f"Expected the Cleveland file with shape (303, 14); got {df.shape}.")
if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in df.dtypes):
    raise ValueError("Unexpected non-numeric source values; review the raw file.")
if np.isinf(df.to_numpy()).any():
    raise ValueError("Unexpected infinite values in the source.")

source_sha256 = sha256(raw_path.read_bytes()).hexdigest()
print(f"Source: {DATA_URL}")
print(f"Local input: {raw_path.relative_to(PROJECT_ROOT)}")
print(f"SHA-256: {source_sha256}")
print(f"Loaded shape: {df.shape}")

Source: https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data
Local input: data\raw\processed.cleveland.data
SHA-256: a74b7efa387bc9d108d7d0115d831fe9b414b29ae7124f331b622b4efa0427c8
Loaded shape: (303, 14)


## 3. Prepare the target and remove it from the features

UCI defines `num = 0` as disease absent and `num = 1, 2, 3, 4` as disease
present. We use that documented grouping for a binary research target called
`heart_disease`. It describes **recorded disease status, not future onset**.

**Code explanation:** First reject missing or unexpected diagnosis values;
otherwise a missing diagnosis could silently become a negative label in a
greater-than comparison. Preserve the original codes separately, map positive
codes to integer 1, and make `X` by dropping `num`. Assert feature/label alignment
and verify that neither the original diagnosis nor the new target is in `X`.
No target values are imputed and no diagnosis information is passed to a transformer.

In [3]:
if df["num"].isna().any() or not df["num"].isin([0, 1, 2, 3, 4]).all():
    raise ValueError("The diagnosis must be non-missing and use UCI codes 0 through 4.")

original_diagnosis = df["num"].copy()
y = original_diagnosis.gt(0).astype("int64").rename("heart_disease")
X = df.drop(columns="num").copy()

assert X.shape[1] == 13
assert X.index.equals(y.index)
assert "num" not in X.columns and "heart_disease" not in X.columns
assert set(y.unique()) == {0, 1}
print("Target mapping: num 0 -> 0 (absent); num 1-4 -> 1 (present)")
print(f"Features: {X.shape}; target: {y.shape}")

Target mapping: num 0 -> 0 (absent); num 1-4 -> 1 (present)
Features: (303, 13); target: (303,)


## 4. Split before fitting any transformation

**Code explanation:** Reserve 20% for testing and use 80% for training. The fixed
seed `42` makes the split repeatable for this file and software environment.
`stratify=y` keeps the binary class proportions approximately equal in both sets;
labels are used here only to allocate the split. The integer sample counts are
242 training records and 61 test records because 20% of 303 is not an integer.

Keep original row indices so each feature row remains traceable to its label.
Assertions check that the two subsets are disjoint, cover all records, and retain
alignment. The class table audits allocation; it is not used to tune preprocessing.
No imputer, scaler, or encoder has been fitted yet.

The training diagnosis table also shows the original UCI codes and their binary mapping. It inspects training labels only; the test labels remain reserved for the split audit and later final evaluation.

In [4]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

assert X_train.index.intersection(X_test.index).empty
assert set(X_train.index) | set(X_test.index) == set(X.index)
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

split_summary = pd.DataFrame({
    "records": [len(X_train), len(X_test)],
    "absent (0)": [int(y_train.eq(0).sum()), int(y_test.eq(0).sum())],
    "present (1)": [int(y_train.eq(1).sum()), int(y_test.eq(1).sum())],
    "present percent": [round(y_train.mean() * 100, 2), round(y_test.mean() * 100, 2)],
}, index=["train", "test"])
display(split_summary)

training_diagnosis_counts = (
    original_diagnosis.loc[X_train.index].astype("int64")
    .value_counts().reindex(range(5), fill_value=0).sort_index()
)
training_target_audit = pd.DataFrame({
    "training_records": training_diagnosis_counts,
    "binary_target": [0, 1, 1, 1, 1],
}).rename_axis("original_num")
print("Original diagnosis codes in the training split:")
display(training_target_audit)
assert int(training_target_audit["training_records"].sum()) == len(y_train)
assert int(training_target_audit.loc[1:, "training_records"].sum()) == int(y_train.sum())

,records,absent (0),present (1),present percent
train,242,131,111,45.87
test,61,33,28,45.90


Original diagnosis codes in the training split:


,training_records,binary_target
original_num,,
0,131,0
1,47,1
2,28,1
3,25,1
4,11,1


## 5. Choose feature roles and inspect training missingness

**Code explanation:** Choose roles from the UCI definitions rather than from
pandas storage types. Five measurement features get median imputation and
standardization. The eight coded/discrete features get most-frequent imputation
and one-hot encoding. We treat `ca` as a discrete category for this baseline,
avoiding a fixed linear effect per additional vessel; `slope` is also encoded
without imposing equal spacing. These are explicit baseline choices, not
choices selected by test performance.

Check that every feature has exactly one role. Inspect missingness using only
`X_train` and stop if a training column is entirely missing, since it would have
no training median or mode. Retain every record. No global mean, median, mode,
or category vocabulary is computed for fitting.

The inspection table shows the raw storage type and intended feature role beside missing-value counts. Floating-point storage can still represent a category, such as chest-pain code `cp`. Preview the first five raw training rows to compare their original values with the prepared tables later.

In [5]:
NUMERIC_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

assert not set(NUMERIC_FEATURES) & set(CATEGORICAL_FEATURES)
assert set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set(X_train.columns)
if X_train.isna().all().any():
    raise ValueError("A training feature is entirely missing; review its preprocessing policy.")

training_missingness = pd.DataFrame({
    "raw_dtype": X_train.dtypes.astype(str),
    "feature_role": [
        "measurement" if column in NUMERIC_FEATURES else "coded/discrete"
        for column in X_train.columns
    ],
    "training_missing_count": X_train.isna().sum(),
    "training_missing_percent": (X_train.isna().mean() * 100).round(2),
    "imputation": [
        "median" if column in NUMERIC_FEATURES else "most_frequent"
        for column in X_train.columns
    ],
})
display(training_missingness)
print("Raw training feature preview:")
display(X_train.head())

,raw_dtype,feature_role,training_missing_count,training_missing_percent,imputation
age,float64,measurement,0,0.00,median
sex,float64,coded/discrete,0,0.00,most_frequent
cp,float64,coded/discrete,0,0.00,most_frequent
trestbps,float64,measurement,0,0.00,median
chol,float64,measurement,0,0.00,median
fbs,float64,coded/discrete,0,0.00,most_frequent
restecg,float64,coded/discrete,0,0.00,most_frequent
thalach,float64,measurement,0,0.00,median
exang,float64,coded/discrete,0,0.00,most_frequent
oldpeak,float64,measurement,0,0.00,median


Raw training feature preview:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
180,48.0,1.0,4.0,124.0,274.0,0.0,2.0,166.0,0.0,0.5,2.0,0.0,7.0
208,55.0,1.0,2.0,130.0,262.0,0.0,0.0,155.0,0.0,0.0,1.0,0.0,3.0
167,54.0,0.0,2.0,132.0,288.0,1.0,2.0,159.0,1.0,0.0,1.0,1.0,3.0
105,54.0,1.0,2.0,108.0,309.0,0.0,0.0,156.0,0.0,0.0,1.0,0.0,7.0
297,57.0,0.0,4.0,140.0,241.0,0.0,0.0,123.0,1.0,0.2,2.0,0.0,7.0


## 6. Build an unfitted preprocessing pipeline

| Feature branch | Ordered steps | Information learned later from training data |
| --- | --- | --- |
| Measurements | Median imputation, then `StandardScaler` | Medians, means, and population standard deviations |
| Coded/discrete values | Most-frequent imputation, then `OneHotEncoder` | Modes and category vocabulary |

**Code explanation:** Each `Pipeline` enforces the branch's step order.
`ColumnTransformer` applies those branches to named columns and joins the
results. `remainder="drop"` explicitly excludes anything outside the feature
lists. The outer pipeline is a reusable preprocessing template with no estimator
for prediction. Construction alone learns no values.

[SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
learns replacement values during fitting; for a tied mode, it picks the smallest
value. The [encoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
keeps all learned categories (`drop=None`) and returns a small dense array for
readability. `handle_unknown="ignore"` represents an unseen category by all zeros
in its feature block, without adding a test-derived column. This avoids a shape
error; it does not make an unknown category clinically meaningful.

In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop=None)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
preprocessing_pipeline = Pipeline(steps=[("preprocessor", preprocessor)])
print(preprocessing_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncod

## 7. Fit and transform the training features only

**Code explanation:** Clone the template and call `fit_transform(X_train)` on
that copy. This is the only fit call in the notebook: it learns preprocessing
statistics, not a prediction model. No labels are needed. Keep the original
template unfitted for future composition with a model.

Access each fitted branch and show the learned imputation values. Snapshot the
fitted arrays and category lists so we can verify that transforming the test
set does not change them. The scaler's means and scales are learned after
median imputation, using training records only.

In [7]:
fitted_preprocessing = clone(preprocessing_pipeline)
X_train_array = fitted_preprocessing.fit_transform(X_train)

fitted_columns = fitted_preprocessing.named_steps["preprocessor"]
fitted_numeric = fitted_columns.named_transformers_["numeric"]
fitted_categorical = fitted_columns.named_transformers_["categorical"]
numeric_imputer = fitted_numeric.named_steps["imputer"]
scaler = fitted_numeric.named_steps["scaler"]
categorical_imputer = fitted_categorical.named_steps["imputer"]
encoder = fitted_categorical.named_steps["encoder"]

learned_before_test = {
    "numeric_fill": numeric_imputer.statistics_.copy(),
    "categorical_fill": categorical_imputer.statistics_.copy(),
    "mean": scaler.mean_.copy(),
    "scale": scaler.scale_.copy(),
    "variance": scaler.var_.copy(),
    "samples_seen": np.array(scaler.n_samples_seen_, copy=True),
    "categories": [values.copy() for values in encoder.categories_],
}
display(pd.concat([
    pd.DataFrame({"strategy": "median", "training_fill_value": numeric_imputer.statistics_}, index=NUMERIC_FEATURES),
    pd.DataFrame({"strategy": "most_frequent", "training_fill_value": categorical_imputer.statistics_}, index=CATEGORICAL_FEATURES),
]).rename_axis("feature"))
print(f"Prepared training array shape: {X_train_array.shape}")

,strategy,training_fill_value
feature,,
age,median,56.0
trestbps,median,130.0
chol,median,244.5
thalach,median,153.5
oldpeak,median,0.8
sex,most_frequent,1.0
cp,most_frequent,4.0
fbs,most_frequent,0.0
restecg,most_frequent,0.0


Prepared training array shape: (242, 28)


## 8. Transform the test features without refitting

**Code explanation:** Call only `transform(X_test)` on the same fitted object.
Missing test values use training fill values; scaling uses training means and
scales; encoding uses the training vocabulary. Never call `fit` or
`fit_transform` on the test set. Compare the learned parameters with the
snapshots to confirm this operation leaves them unchanged.

Do not compare test distributions to choose a different imputer, scaling policy,
or feature set. Such decisions would use holdout information even if the code
fitted each transformer on the correct rows.

In [8]:
X_test_array = fitted_preprocessing.transform(X_test)

for key, current in [
    ("numeric_fill", numeric_imputer.statistics_),
    ("categorical_fill", categorical_imputer.statistics_),
    ("mean", scaler.mean_),
    ("scale", scaler.scale_),
    ("variance", scaler.var_),
    ("samples_seen", np.asarray(scaler.n_samples_seen_)),
]:
    np.testing.assert_array_equal(learned_before_test[key], current)
for before, after in zip(learned_before_test["categories"], encoder.categories_):
    np.testing.assert_array_equal(before, after)

print(f"Prepared test array shape: {X_test_array.shape}")
print("Test transformation left learned preprocessing parameters unchanged.")

Prepared test array shape: (61, 28)
Test transformation left learned preprocessing parameters unchanged.


## 9. Verify the fitted values against training data

**Code explanation:** Independently compute medians and modes from `X_train`
and compare them with the fitted imputers. Compute means and population
variances after filling training measurements and compare with the scaler.
Check that its sample count equals the training size. Compare each learned
vocabulary to the imputed training categories. These checks would expose many
accidental full-dataset fits and confirm what the pipeline actually learned.
They do not recompute anything from test values or refit any component.

In [9]:
expected_medians = X_train[NUMERIC_FEATURES].median()
expected_modes = X_train[CATEGORICAL_FEATURES].mode().iloc[0]
np.testing.assert_allclose(numeric_imputer.statistics_, expected_medians.to_numpy())
np.testing.assert_allclose(categorical_imputer.statistics_, expected_modes.to_numpy())

imputed_training_numeric = X_train[NUMERIC_FEATURES].fillna(expected_medians)
np.testing.assert_allclose(scaler.mean_, imputed_training_numeric.mean().to_numpy())
np.testing.assert_allclose(scaler.var_, imputed_training_numeric.var(ddof=0).to_numpy())
assert np.all(np.asarray(scaler.n_samples_seen_) == len(X_train))

imputed_training_categories = X_train[CATEGORICAL_FEATURES].fillna(expected_modes)
for column, learned_categories in zip(CATEGORICAL_FEATURES, encoder.categories_):
    np.testing.assert_array_equal(
        learned_categories, np.sort(imputed_training_categories[column].unique()),
    )
assert not hasattr(preprocessing_pipeline.named_steps["preprocessor"], "transformers_")
print("Verified: imputers, scaler, and category vocabularies match training data only.")
print("The reusable preprocessing template is still unfitted.")

Verified: imputers, scaler, and category vocabularies match training data only.
The reusable preprocessing template is still unfitted.


## 10. Label and validate the prepared feature tables

**Code explanation:** Obtain feature names from the fitted pipeline and wrap
its arrays in dataframes with the original split indices. Verify that both
outputs have the same columns, the expected number of records, finite values,
no missing cells, and indices aligned with their targets. The explicit feature
lists contain neither `num` nor `heart_disease`.

Preview training rows only. The numerical columns are standardized while each
one-hot column indicates a learned category. A zero in a standardized column
means the training mean, not an original measurement of zero. Only training
numerical columns should have means near zero; test means need not be zero.

In [10]:
feature_names = fitted_preprocessing.get_feature_names_out()
X_train_prepared = pd.DataFrame(X_train_array, index=X_train.index, columns=feature_names)
X_test_prepared = pd.DataFrame(X_test_array, index=X_test.index, columns=feature_names)

assert X_train_prepared.columns.equals(X_test_prepared.columns)
assert X_train_prepared.index.equals(y_train.index)
assert X_test_prepared.index.equals(y_test.index)
assert len(X_train_prepared) == len(X_train)
assert len(X_test_prepared) == len(X_test)
for prepared in [X_train_prepared, X_test_prepared]:
    assert not prepared.isna().any().any()
    assert np.isfinite(prepared.to_numpy()).all()
np.testing.assert_allclose(
    X_train_prepared.loc[:, [f"numeric__{column}" for column in NUMERIC_FEATURES]].mean(),
    0, atol=1e-12,
)
print(f"Training: {X_train_prepared.shape}; testing: {X_test_prepared.shape}")
print("Prepared features:", feature_names.tolist())
display(X_train_prepared.head())

Training: (242, 28); testing: (61, 28)
Prepared features: ['numeric__age', 'numeric__trestbps', 'numeric__chol', 'numeric__thalach', 'numeric__oldpeak', 'categorical__sex_0.0', 'categorical__sex_1.0', 'categorical__cp_1.0', 'categorical__cp_2.0', 'categorical__cp_3.0', 'categorical__cp_4.0', 'categorical__fbs_0.0', 'categorical__fbs_1.0', 'categorical__restecg_0.0', 'categorical__restecg_1.0', 'categorical__restecg_2.0', 'categorical__exang_0.0', 'categorical__exang_1.0', 'categorical__slope_1.0', 'categorical__slope_2.0', 'categorical__slope_3.0', 'categorical__ca_0.0', 'categorical__ca_1.0', 'categorical__ca_2.0', 'categorical__ca_3.0', 'categorical__thal_3.0', 'categorical__thal_6.0', 'categorical__thal_7.0']


,numeric__age,numeric__trestbps,numeric__chol,numeric__thalach,numeric__oldpeak,categorical__sex_0.0,categorical__sex_1.0,categorical__cp_1.0,categorical__cp_2.0,categorical__cp_3.0,...,categorical__slope_1.0,categorical__slope_2.0,categorical__slope_3.0,categorical__ca_0.0,categorical__ca_1.0,categorical__ca_2.0,categorical__ca_3.0,categorical__thal_3.0,categorical__thal_6.0,categorical__thal_7.0
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 11. Check missing and unseen values on a synthetic input

**Code explanation:** Copy one training row and introduce missing `age` and
`thal` values plus an artificial chest-pain code `999`. This synthetic record
tests the preprocessing contract without changing either real split. Transform
it using the fitted pipeline, then verify finite results, median-based age
replacement, mode-based `thal` replacement, and an all-zero chest-pain block
for the unseen code. The vocabulary and output width must stay fixed.

This is a technical check, not a valid clinical record. In a future application,
input validation should flag invalid codes even though the encoder can handle
them without failing.

In [11]:
probe = X_train.iloc[[0]].copy()
probe.loc[:, "age"] = np.nan
probe.loc[:, "thal"] = np.nan
probe.loc[:, "cp"] = 999.0
probe_array = fitted_preprocessing.transform(probe)
probe_prepared = pd.DataFrame(probe_array, columns=feature_names)

assert probe_array.shape == (1, len(feature_names))
assert np.isfinite(probe_array).all()
age_index = NUMERIC_FEATURES.index("age")
expected_age = (numeric_imputer.statistics_[age_index] - scaler.mean_[age_index]) / scaler.scale_[age_index]
np.testing.assert_allclose(probe_prepared["numeric__age"].iloc[0], expected_age)
cp_columns = [name for name in feature_names if name.startswith("categorical__cp_")]
assert probe_prepared[cp_columns].eq(0).all().all()

imputed_probe_categories = categorical_imputer.transform(probe[CATEGORICAL_FEATURES])
thal_index = CATEGORICAL_FEATURES.index("thal")
np.testing.assert_equal(
    imputed_probe_categories[0, thal_index], categorical_imputer.statistics_[thal_index],
)
for before, after in zip(learned_before_test["categories"], encoder.categories_):
    np.testing.assert_array_equal(before, after)
print("Synthetic checks passed: missing values handled; unseen code preserved the feature schema.")

Synthetic checks passed: missing values handled; unseen code preserved the feature schema.


## What is ready for the next stage

| In-memory object | Purpose |
| --- | --- |
| `X_train`, `y_train` | Raw training features and binary labels |
| `X_test`, `y_test` | Held-out raw features and labels |
| `preprocessing_pipeline` | Unfitted template for a future combined preprocessing/model pipeline |
| `fitted_preprocessing` | Transformers fitted on training features only |
| `X_train_prepared`, `X_test_prepared` | Aligned, finite feature tables for this fixed split |

The source file is unchanged. No datasets or fitted objects are exported, and
no classifier, predictions, performance metrics, or frontend are created.
Notebook outputs retain the audit so the process can be reviewed.

## Keep the boundary intact in future work

- For cross-validation or tuning, combine a fresh clone of
  `preprocessing_pipeline` with the future estimator and pass **raw training
  features** to cross-validation. Each fold must learn its own preprocessing.
  Using `X_train_prepared` for cross-validation would expose each validation
  fold to statistics learned from the entire training split.
- Keep this test split and seed fixed. Use training folds for development and
  reserve the holdout for final evaluation rather than repeated choices.
- The earlier exploration notebook examined the full cohort. Training-only
  transformer fitting prevents preprocessing leakage, but this holdout is not
  an independently untouched external validation cohort.
- The source lacks usable patient grouping and chronology in these 14 columns;
  this is a reproducible row-based research split. Clinical features such as
  `ca` and `thal` include diagnostic test information. A later prediction use
  case must specify which features are available at its intended decision time.

Preprocessing checks establish correct separation for this workflow; they do
not establish clinical validity. This notebook stops before model training.